# WET-004: Donut Deblending Test

Owner: **Bryce Kalmbach** <br>
Last Verified to Run: **2025-04-08** <br>
Software Version:
  - `ts_wep`: **14.1.1**
  - `donut_viz`: **1.6.2**
  - `lsst_distrib`: **w_2025_13**

**Goal**: Demonstrate tooling available to measure how well the WEP Zernike Estimation performs in the presence of blended objects by comparing the results to average Zernike estimates for the ComCam raft.

In [ ]:
# Times Square Parameters
collection_name = 'u/brycek/aosRefitWcs_danish_singleBlends_80pxMinSep'
detector = 0
min_seq_num = 90
max_seq_num = 107
day_obs = 20241115

In [ ]:
import numpy as np
from copy import copy
from astropy.visualization import ZScaleInterval
from matplotlib import pyplot as plt
from lsst.daf.butler import Butler
from lsst.ts.wep.utils import getPsfGradPerZernike
%matplotlib inline

In [ ]:
butler = Butler('/repo/main')

In [ ]:
camera = butler.get('camera', {'instrument': "LSSTComCam"}, collections=collection_name)

## Identify Blended Donuts

In [ ]:
refs_with_visit_tables = butler.query_datasets('aggregateAOSVisitTableRaw', 
                                             collections=collection_name,
                                             where=f"exposure.day_obs = {day_obs} and exposure.seq_num >= {min_seq_num} and exposure.seq_num <= {max_seq_num} and instrument = 'LSSTComCam'")

In [ ]:
ref_on = 0
dt = butler.get(refs_with_visit_tables[ref_on])
dataId = refs_with_visit_tables[ref_on].to_simple().dataId.dataId
print(f"Using Visit Number: {dataId['visit']}")

In [ ]:
# Find donuts with blended centroid information
blended_idx = list()
blend_x = list()
blend_y = list()
for idx, loc in enumerate(dt.meta['blendInfo']['blend_centroid_x_extra']):
    if len(loc) > 0:
        blended_idx.append(idx)
        blend_x.append(dt.meta['blendInfo']['blend_centroid_x_extra'][idx][0])
        blend_y.append(dt.meta['blendInfo']['blend_centroid_y_extra'][idx][0])
blend_x = np.array(blend_x)
blend_y = np.array(blend_y)

In [ ]:
# Calculate the separation between the centroid of the main object and the blended object in pixels
dt_blended = dt[blended_idx]
dt_blended['blend_centroid_x'] = blend_x
dt_blended['blend_centroid_y'] = blend_y
separation = np.sqrt((dt_blended['blend_centroid_x'] - dt_blended['centroid_x'])**2 + (dt_blended['blend_centroid_y'] - dt_blended['centroid_y'])**2)
dt_blended['separation'] = separation
separation_cut = separation <= 160.
dt_blended = dt_blended[separation_cut]
blended_idx = [blended_idx[keep_idx] for keep_idx, x in enumerate(separation_cut) if x == True]

In [ ]:
dt_blended

In [ ]:
dt_blended['separation'], blended_idx

## View Blended Objects

In [ ]:
ds_extra_visit = butler.get('donutStampsExtraVisit', dataId=dataId, collections=collection_name)
ds_intra_visit = butler.get('donutStampsIntraVisit', dataId=dataId, collections=collection_name)

In [ ]:
use_idx = blended_idx[0]

In [ ]:
fig = plt.figure(figsize=(12, 6))
fig.add_subplot(1,2,1)
plt.imshow(ds_extra_visit[use_idx].stamp_im.image.array, vmax=500)
fig.add_subplot(1,2,2)
plt.imshow(ds_extra_visit[use_idx].stamp_im.mask.array)

In [ ]:
fig = plt.figure(figsize=(12, 6))
fig.add_subplot(1,2,1)
plt.imshow(ds_intra_visit[use_idx].stamp_im.image.array, vmax=500)
fig.add_subplot(1,2,2)
plt.imshow(ds_intra_visit[use_idx].stamp_im.mask.array)

## Evaluate Zernike estimation performance on blended objects

In [ ]:
noll_indices = dt.meta['nollIndices']

noll_max = np.max(noll_indices)
noll_min = np.min(noll_indices)
conv_array = getPsfGradPerZernike(jmin=noll_min, jmax=noll_max)

In [ ]:
def convertZernToArcsec(row_zern, noll_idx=noll_indices, psf_convert_array=conv_array):
    """Convert the Zernikes from microns to arcseconds.
    
    Parameters
    ----------
    row_zern
        Zernike array from row in visit table.
    noll_idx
        Numbers of the noll indices used in zernike calculation.
    psf_convert_array
        Array for converting microns -> arcseconds.
        
    Returns
    -------
    numpy.ndarray
        Numpy array with zernike output in arcseconds with zeros for zernike values not included in noll_idx.
    """
    noll_max = np.max(noll_idx)
    noll_min = np.min(noll_idx)
    zk_array = np.zeros((noll_max - noll_min + 1))
    zk_array[noll_idx - noll_min - 1] = row_zern
    zk_array = zk_array * psf_convert_array
    return zk_array

In [ ]:
def calcAverageZern(dataId, noll_idx=noll_indices, psf_convert_array=conv_array):
    """Provide the average zernike for the visit on the comcam raft.
    
    Parameters
    ----------
    dataId
        dataId we can use to get the average zernike values.
    noll_idx
        Numbers of the noll indices used in zernike calculation.
    psf_convert_array
        Array for converting microns -> arcseconds.
        
    Returns
    -------
    numpy.ndarray
        Numpy array with average zernike output in arcseconds across comcam detector in arcseconds.
    """
    avg_visit_table = butler.get(
        'aggregateAOSVisitTableAvg', 
        dataId=dataId,
        collections=collection_name,
        )
    noll_max = np.max(noll_idx)
    noll_min = np.min(noll_idx)
    zk_array = np.zeros((noll_max - noll_min + 1))
    for zk_ccs in avg_visit_table['zk_CCS']:
        zk_array += convertZernToArcsec(zk_ccs)
    zk_array /= len(avg_visit_table)
    return zk_array

In [ ]:
label_set = False
avg_zern = calcAverageZern(dataId)
for idx in range(len(dt_blended)):
    if label_set is False:
        plt.plot(noll_indices, convertZernToArcsec(dt_blended[idx]['zk_CCS'])[noll_indices - noll_min], c='C0', label='Blended objects')
        label_set = True
    else:
        plt.plot(noll_indices, dt_blended[idx]['zk_CCS'], c='C0')
plt.plot(noll_indices, avg_zern[noll_indices-noll_min], c='C1', label='Average Zernike Estimates')
plt.legend()
plt.xlabel('Noll Index')
plt.ylabel('Zernike Value (arcsec)')

In [ ]:
zern_diff = []
for idx in range(len(dt_blended)):
    zern_diff.append(np.sqrt(np.sum(np.square(avg_zern - convertZernToArcsec(dt_blended[idx]['zk_CCS'], noll_indices)))))
plt.scatter(dt_blended['separation'], zern_diff)
plt.ylabel('Deviation from Avg Zernike Wavefront (arcsec)')
plt.xlabel('Blended Centroid Separation (pixels)')

Not many bright blended objects in the test field but the tooling has been developed to quickly analyze with LSSTCam.